# AI Knowledge Assistant with Memory — RAG over PDF & Google Sheets

## Project Overview

This project implements a basic AI Knowledge Assistant that combines **Retrieval-Augmented Generation (RAG)** with **conversation memory**.

The assistant integrates knowledge from two sources:

* **Stanford CS229 PDF lecture notes**
* **Google Sheets structured knowledge records**

The documents are converted into searchable representations using **Hugging Face embeddings** and indexed with **FAISS**. For each user query, the system retrieves relevant knowledge and provides it to a local **Qwen2.5-1.5B-Instruct** model to generate a grounded answer.

The assistant also maintains a **full conversation buffer**, allowing it to understand follow-up questions and references to previous turns.

## Project Objectives

* Integrate knowledge from PDF documents and Google Sheets.
* Convert heterogeneous sources into structured LangChain documents.
* Generate semantic embeddings using `all-MiniLM-L6-v2`.
* Build a unified FAISS vector database.
* Implement a Retrieval-Augmented Generation pipeline.
* Integrate full conversation memory for multi-turn interactions.
* Evaluate retrieval quality, answer generation, and out-of-scope behavior.
* Demonstrate a reproducible end-to-end local RAG workflow.

## Main Technologies

* Python
* LangChain
* Hugging Face Transformers
* Sentence Transformers
* FAISS
* PyPDF
* Google Sheets / gspread
* Qwen2.5-1.5B-Instruct
* Google Colab / NVIDIA Tesla T4


## Project Architecture

The system follows an end-to-end **Retrieval-Augmented Generation (RAG)** architecture with multi-turn conversation memory.

```text
                         USER QUESTION
                              |
                              v
                    +-------------------+
                    |  Query Processing |
                    +-------------------+
                              |
                              v
                    +-------------------+
                    | Semantic Retrieval|
                    |      FAISS        |
                    +-------------------+
                              |
                +-------------+-------------+
                |                           |
                v                           v
        PDF Knowledge Base         Google Sheets Knowledge
        14 PDFs / 157 pages        50 structured records
                |                           |
                +-------------+-------------+
                              |
                              v
                    Retrieved Context
                              |
                 +------------+------------+
                 |                         |
                 v                         v
       Conversation Memory         Retrieved Knowledge
       Full Buffer History         Top-K Documents
                 |                         |
                 +------------+------------+
                              |
                              v
                   Qwen2.5-1.5B-Instruct
                              |
                              v
                     Grounded Answer
                              |
                              v
                            USER
```

### Pipeline Components

| Component           | Implementation                           |
| ------------------- | ---------------------------------------- |
| PDF Source          | Stanford CS229 Lecture Notes             |
| Structured Source   | Google Sheets                            |
| Document Processing | PyPDF + LangChain Documents              |
| Text Chunking       | RecursiveCharacterTextSplitter           |
| Embeddings          | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector Database     | FAISS                                    |
| Retrieval           | Top-K Semantic Search                    |
| LLM                 | `Qwen/Qwen2.5-1.5B-Instruct`             |
| Memory              | Full Conversation Buffer                 |
| Runtime             | Google Colab + NVIDIA Tesla T4           |

### End-to-End Flow

1. Load knowledge from PDFs and Google Sheets.
2. Convert the sources into structured documents.
3. Split PDF content into overlapping chunks.
4. Generate vector embeddings.
5. Store all knowledge in a unified FAISS index.
6. Retrieve the most relevant documents for each query.
7. Combine retrieved context with conversation history.
8. Generate a grounded response using the local Qwen model.
9. Store the new user question and assistant response in conversation memory.
10. Evaluate retrieval, answer generation, and multi-turn memory behavior.


## 1. Environment Setup

This project was developed and evaluated in **Google Colab** using an NVIDIA Tesla T4 GPU.

The required libraries support:

* PDF document processing
* LangChain document pipelines
* Hugging Face embeddings
* FAISS vector search
* Google Sheets integration
* Local LLM inference


In [ ]:
# ============================================================
# Environment Setup
# ============================================================

!pip -q install \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    gspread \
    google-auth \
    pandas \
    requests \
    transformers \
    accelerate

print("Environment setup completed successfully.")

## 2. Imports & Hardware Verification

The project uses LangChain for document processing and retrieval, Hugging Face models for embeddings and generation, FAISS for vector search, and Google APIs for structured knowledge integration.

The runtime hardware is also verified to ensure that GPU acceleration is available for embedding and local LLM inference.


In [37]:
# ============================================================
# Imports & Hardware Verification
# ============================================================

import os
import torch
import pandas as pd
import gspread
import google.auth

from pypdf import PdfReader

from google.colab import auth

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from transformers import AutoTokenizer, AutoModelForCausalLM


# ------------------------------------------------------------
# Hardware Verification
# ------------------------------------------------------------

print("=" * 80)
print("ENVIRONMENT VERIFICATION")
print("=" * 80)

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("GPU: CPU only")

print("\nEnvironment verification completed.")

ENVIRONMENT VERIFICATION

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8

Environment verification completed.


## 3. PDF Knowledge Base

The primary unstructured knowledge source consists of **Stanford CS229 lecture notes**.

The project uses 14 selected PDF files from the CS229 notes archive. The documents are downloaded, validated, and then converted into LangChain `Document` objects while preserving the original filename and page number as metadata.


In [38]:
# ============================================================
# PDF Knowledge Base
# ============================================================

import requests

BASE_URL = "https://cs229.stanford.edu/notes_archive/"
PDF_DIR = "/content/cs229_pdfs"

core_pdfs = [
    "cs229-notes1.pdf",
    "cs229-notes2.pdf",
    "cs229-notes3.pdf",
    "cs229-notes4.pdf",
    "cs229-notes5.pdf",
    "cs229-notes6.pdf",
    "cs229-notes7a.pdf",
    "cs229-notes7b.pdf",
    "cs229-notes8.pdf",
    "cs229-notes9.pdf",
    "cs229-notes10.pdf",
    "cs229-notes11.pdf",
    "cs229-notes12.pdf",
    "cs229-notes13.pdf",
]

os.makedirs(PDF_DIR, exist_ok=True)

downloaded_files = []

for filename in core_pdfs:
    filepath = os.path.join(PDF_DIR, filename)

    response = requests.get(
        BASE_URL + filename,
        timeout=30
    )

    response.raise_for_status()

    with open(filepath, "wb") as f:
        f.write(response.content)

    downloaded_files.append(filepath)

print("=" * 80)
print("PDF DOWNLOAD COMPLETED")
print("=" * 80)

print(f"\nPDF files downloaded: {len(downloaded_files)}")
print(f"Storage directory: {PDF_DIR}")

PDF DOWNLOAD COMPLETED

PDF files downloaded: 14
Storage directory: /content/cs229_pdfs


## 4. PDF Validation & Text Extraction

Each downloaded PDF is validated by reading its pages with `PyPDF`.
The extracted text is converted into LangChain `Document` objects while preserving the source filename, page number, and source type as metadata.


In [39]:
# ============================================================
# PDF Validation & Text Extraction
# ============================================================

documents = []
pdf_validation = []

for filepath in downloaded_files:
    filename = os.path.basename(filepath)
    reader = PdfReader(filepath)

    page_count = len(reader.pages)
    word_count = 0

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()

        word_count += len(text.split())

        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": filename,
                        "page": page_number,
                        "source_type": "PDF"
                    }
                )
            )

    pdf_validation.append({
        "file": filename,
        "pages": page_count,
        "words": word_count,
        "status": "OK"
    })


validation_df = pd.DataFrame(pdf_validation)

print("=" * 80)
print("PDF VALIDATION COMPLETED")
print("=" * 80)

print(f"\nTotal PDF files: {len(validation_df)}")
print(f"Total pages: {validation_df['pages'].sum():,}")
print(f"Total extracted words: {validation_df['words'].sum():,}")
print(f"LangChain documents created: {len(documents):,}")

print("\nValidation status:")
print(validation_df["status"].value_counts().to_dict())

print("\nFirst document metadata:")
print(documents[0].metadata)

print("\nFirst document preview:")
print(documents[0].page_content[:500])

PDF VALIDATION COMPLETED

Total PDF files: 14
Total pages: 157
Total extracted words: 49,041
LangChain documents created: 157

Validation status:
{'OK': 14}

First document metadata:
{'source': 'cs229-notes1.pdf', 'page': 1, 'source_type': 'PDF'}

First document preview:
CS229 Lecture notes
Andrew Ng
Supervised learning
Let’s start by talking about a few examples of supervised lea rning problems.
Suppose we have a dataset giving the living areas and prices o f 47 houses
from Portland, Oregon:
Living area (feet 2) Price (1000$s)
2104 400
1600 330
2400 369
1416 232
3000 540
... ...
We can plot this data:
500 1000 1500 2000 2500 3000 3500 4000 4500 5000
0
100
200
300
400
500
600
700
800
900
1000
housing prices
square feet
price (in $1000)
Given data like this, how 


## 5. Text Chunking

The extracted PDF documents are split into smaller overlapping chunks using `RecursiveCharacterTextSplitter`.

Chunking improves semantic retrieval by allowing the vector database to match user questions against focused sections of the source material while preserving contextual continuity through overlapping text.


In [40]:
# ============================================================
# Text Chunking
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(documents)

print("=" * 80)
print("TEXT CHUNKING COMPLETED")
print("=" * 80)

print(f"\nOriginal documents: {len(documents):,}")
print(f"Total chunks: {len(chunks):,}")

if chunks:
    print(f"First chunk length: {len(chunks[0].page_content):,} characters")
    print("\nFirst chunk metadata:")
    print(chunks[0].metadata)

    print("\nFirst chunk preview:")
    print(chunks[0].page_content[:500])

TEXT CHUNKING COMPLETED

Original documents: 157
Total chunks: 355
First chunk length: 614 characters

First chunk metadata:
{'source': 'cs229-notes1.pdf', 'page': 1, 'source_type': 'PDF'}

First chunk preview:
CS229 Lecture notes
Andrew Ng
Supervised learning
Let’s start by talking about a few examples of supervised lea rning problems.
Suppose we have a dataset giving the living areas and prices o f 47 houses
from Portland, Oregon:
Living area (feet 2) Price (1000$s)
2104 400
1600 330
2400 369
1416 232
3000 540
... ...
We can plot this data:
500 1000 1500 2000 2500 3000 3500 4000 4500 5000
0
100
200
300
400
500
600
700
800
900
1000
housing prices
square feet
price (in $1000)
Given data like this, how 


## 6. Google Sheets Knowledge Integration

The second knowledge source is a structured Google Sheet containing machine learning concepts and metadata.

Google Colab authentication is used to securely access the spreadsheet through the Google Sheets API. The structured records are later converted into LangChain `Document` objects and combined with the PDF knowledge base.


In [41]:
# ============================================================
# Google Sheets Authentication
# ============================================================

auth.authenticate_user()

credentials, project_id = google.auth.default(
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
)

gc = gspread.authorize(credentials)

print("=" * 80)
print("GOOGLE SHEETS AUTHENTICATION")
print("=" * 80)

print("\nAuthentication completed successfully.")
print("Google Sheets client is ready.")

GOOGLE SHEETS AUTHENTICATION

Authentication completed successfully.
Google Sheets client is ready.


## 7. Load Structured Google Sheet

The structured knowledge base is stored in a Google Sheet with machine learning concepts, descriptions, key concepts, source references, and difficulty levels.

The existing spreadsheet is loaded through its ID and converted into a pandas DataFrame for validation and downstream document processing.


In [42]:
# ============================================================
# Load Structured Google Sheet
# ============================================================

SPREADSHEET_ID = "1ywDVgK5ywkrTzRHzbaVFFSy24zbRQboCjap4WEdgjgA"
WORKSHEET_NAME = "Knowledge"

spreadsheet = gc.open_by_key(SPREADSHEET_ID)
worksheet = spreadsheet.worksheet(WORKSHEET_NAME)

records = worksheet.get_all_records()
sheet_df = pd.DataFrame(records)

print("=" * 80)
print("GOOGLE SHEETS DATA LOADED")
print("=" * 80)

print(f"\nSpreadsheet: {spreadsheet.title}")
print(f"Worksheet: {worksheet.title}")
print(f"Rows: {len(sheet_df):,}")
print(f"Columns: {len(sheet_df.columns):,}")

print("\nColumns:")
print(list(sheet_df.columns))

print("\nFirst 3 records:")
display(sheet_df.head(3))

GOOGLE SHEETS DATA LOADED

Spreadsheet: AI Knowledge Assistant - Structured Knowledge
Worksheet: Knowledge
Rows: 50
Columns: 7

Columns:
['ID', 'Topic', 'Category', 'Description', 'Key Concepts', 'Related PDF', 'Level']

First 3 records:


,ID,Topic,Category,Description,Key Concepts,Related PDF,Level
0,ML-001,Supervised Learning,Machine Learning,Learning from labeled examples to predict outp...,"Regression, Classification, Training Set, Hypo...",cs229-notes1.pdf,Beginner
1,ML-002,Linear Regression,Machine Learning,A supervised learning method for predicting co...,"Least Squares, Cost Function, Gradient Descent",cs229-notes1.pdf,Beginner
2,ML-003,Logistic Regression,Machine Learning,A classification algorithm that models the pro...,"Classification, Sigmoid Function, Decision Bou...",cs229-notes2.pdf,Beginner


## 8. Structured Knowledge Documents

Each Google Sheets record is converted into a LangChain `Document`.

The structured fields are combined into a searchable text representation, while metadata such as the row ID, topic, difficulty level, and source type is preserved for retrieval and source tracking.


In [43]:
# ============================================================
# Convert Google Sheets Records to LangChain Documents
# ============================================================

sheet_documents = []

for _, row in sheet_df.iterrows():
    structured_text = (
        f"Topic: {row['Topic']}\n"
        f"Category: {row['Category']}\n"
        f"Description: {row['Description']}\n"
        f"Key Concepts: {row['Key Concepts']}\n"
        f"Related PDF: {row['Related PDF']}\n"
        f"Level: {row['Level']}"
    )

    sheet_documents.append(
        Document(
            page_content=structured_text,
            metadata={
                "source": "AI Knowledge Assistant - Structured Knowledge",
                "sheet": WORKSHEET_NAME,
                "row_id": row["ID"],
                "source_type": "Google Sheets",
                "topic": row["Topic"],
                "level": row["Level"]
            }
        )
    )

print("=" * 80)
print("GOOGLE SHEETS DOCUMENT CONVERSION")
print("=" * 80)

print(f"\nGoogle Sheets records: {len(sheet_df):,}")
print(f"LangChain documents created: {len(sheet_documents):,}")

print("\nFirst document metadata:")
print(sheet_documents[0].metadata)

print("\nFirst document preview:")
print(sheet_documents[0].page_content)

GOOGLE SHEETS DOCUMENT CONVERSION

Google Sheets records: 50
LangChain documents created: 50

First document metadata:
{'source': 'AI Knowledge Assistant - Structured Knowledge', 'sheet': 'Knowledge', 'row_id': 'ML-001', 'source_type': 'Google Sheets', 'topic': 'Supervised Learning', 'level': 'Beginner'}

First document preview:
Topic: Supervised Learning
Category: Machine Learning
Description: Learning from labeled examples to predict outputs for new inputs.
Key Concepts: Regression, Classification, Training Set, Hypothesis
Related PDF: cs229-notes1.pdf
Level: Beginner


## 9. Embeddings & Unified Vector Database

The PDF chunks and structured Google Sheets records are combined into a single knowledge base.

Each document is converted into a semantic vector using the `sentence-transformers/all-MiniLM-L6-v2` embedding model. The resulting embeddings are stored in a FAISS vector database to enable efficient semantic similarity search across both knowledge sources.


In [44]:
# ============================================================
# Embeddings & Unified FAISS Vector Database
# ============================================================

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": device
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32
    }
)

# Combine both knowledge sources
all_documents = chunks + sheet_documents

# Build unified FAISS index
unified_vectorstore = FAISS.from_documents(
    documents=all_documents,
    embedding=embeddings
)

# Save the vector database
VECTORSTORE_PATH = "/content/ai_knowledge_assistant_faiss"

unified_vectorstore.save_local(VECTORSTORE_PATH)

print("=" * 80)
print("UNIFIED VECTOR DATABASE CREATED")
print("=" * 80)

print(f"\nEmbedding model: {EMBEDDING_MODEL}")
print(f"Device: {device}")
print(f"PDF chunks: {len(chunks):,}")
print(f"Google Sheets documents: {len(sheet_documents):,}")
print(f"Total indexed documents: {len(all_documents):,}")
print(f"Embedding dimension: {unified_vectorstore.index.d}")
print(f"Vector store: FAISS")
print(f"Saved to: {VECTORSTORE_PATH}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

UNIFIED VECTOR DATABASE CREATED

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda
PDF chunks: 355
Google Sheets documents: 50
Total indexed documents: 405
Embedding dimension: 384
Vector store: FAISS
Saved to: /content/ai_knowledge_assistant_faiss


## 10. Unified Semantic Search Test

The unified FAISS index is tested with representative questions to verify semantic retrieval across both knowledge sources.

The test checks whether relevant information can be retrieved from:

* Stanford CS229 PDF chunks
* Structured Google Sheets records


In [45]:
# ============================================================
# Unified Semantic Search Test
# ============================================================

test_queries = [
    "What is supervised learning?",
    "What is the difference between L1 and L2 regularization?"
]

for query in test_queries:
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    results = unified_vectorstore.similarity_search(
        query,
        k=5
    )

    for rank, doc in enumerate(results, start=1):
        source_type = doc.metadata.get("source_type")

        if source_type == "PDF":
            source_info = (
                f"PDF: {doc.metadata.get('source')}, "
                f"Page: {doc.metadata.get('page')}"
            )

        elif source_type == "Google Sheets":
            source_info = (
                f"Google Sheets: "
                f"Row {doc.metadata.get('row_id')}, "
                f"Topic: {doc.metadata.get('topic')}"
            )

        else:
            source_info = "Unknown source"

        print(f"\nResult {rank}")
        print(f"Source: {source_info}")
        print(f"Content: {doc.page_content[:300]}...")

QUERY: What is supervised learning?

Result 1
Source: Google Sheets: Row ML-001, Topic: Supervised Learning
Content: Topic: Supervised Learning
Category: Machine Learning
Description: Learning from labeled examples to predict outputs for new inputs.
Key Concepts: Regression, Classification, Training Set, Hypothesis
Related PDF: cs229-notes1.pdf
Level: Beginner...

Result 2
Source: PDF: cs229-notes1.pdf, Page: 2
Content: function h is called a hypothesis. Seen pictorially, the process is therefore
like this:
Training 
    set
 house.)
(living area of
Learning 
algorithm
h predicted yx
(predicted price)
of house)
When the target variable that we’re trying to predict is cont inuous, such
as in our housing example, we ...

Result 3
Source: PDF: cs229-notes2.pdf, Page: 1
Content: decision boundary it falls, and makes its prediction accord ingly.
Here’s a diﬀerent approach. First, looking at elephants, we c an build a
model of what elephants look like. Then, looking at dogs, we c an build a


## 11. Local LLM for Answer Generation

A local instruction-tuned language model is used as the generation component of the RAG pipeline.

The project uses **Qwen2.5-1.5B-Instruct**, loaded with Hugging Face Transformers and executed on the available NVIDIA Tesla T4 GPU.


In [46]:
# ============================================================
# Local LLM
# ============================================================

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

print("=" * 80)
print("LOCAL LLM LOADED")
print("=" * 80)

print(f"\nModel: {LLM_MODEL}")
print(f"Device: {llm_model.device}")
print(f"Dtype: {llm_model.dtype}")
print(f"Parameters: {llm_model.num_parameters():,}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LOCAL LLM LOADED

Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Dtype: torch.float16
Parameters: 1,543,714,304


## 12. Retrieval-Augmented Generation Pipeline

The RAG pipeline retrieves the most relevant documents from the unified FAISS index and provides them as context to the local Qwen instruction model.

The model is explicitly instructed to use the retrieved knowledge as the primary factual source and to avoid generating unsupported information.


In [47]:
# ============================================================
# Basic RAG Generation Pipeline
# ============================================================

def generate_rag_answer(
    question,
    vectorstore,
    top_k=5,
    max_new_tokens=250
):
    # --------------------------------------------------------
    # 1. Retrieve relevant documents
    # --------------------------------------------------------
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=top_k
    )

    # --------------------------------------------------------
    # 2. Build retrieved context
    # --------------------------------------------------------
    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        source_type = doc.metadata.get("source_type")

        if source_type == "PDF":
            source_info = (
                f"PDF: {doc.metadata.get('source')}, "
                f"Page: {doc.metadata.get('page')}"
            )

        elif source_type == "Google Sheets":
            source_info = (
                f"Google Sheets: "
                f"Row {doc.metadata.get('row_id')}, "
                f"Topic: {doc.metadata.get('topic')}"
            )

        else:
            source_info = "Unknown source"

        context_parts.append(
            f"[Document {i}]\n"
            f"Source: {source_info}\n"
            f"{doc.page_content}"
        )

    context = "\n\n".join(context_parts)

    # --------------------------------------------------------
    # 3. Define grounded system instructions
    # --------------------------------------------------------
    system_message = """You are an AI Knowledge Assistant.

Answer the user's question using ONLY the provided knowledge base context.

Rules:
1. Do not invent facts that are not supported by the context.
2. If the answer cannot be determined from the context, say:
   "I don't know based on the provided knowledge base."
3. Give a clear and concise answer.
4. When useful, mention the relevant concepts from the context.
"""

    # --------------------------------------------------------
    # 4. Build user prompt
    # --------------------------------------------------------
    user_message = f"""Knowledge Base Context:

{context}

User Question:
{question}

Answer:"""

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    # --------------------------------------------------------
    # 5. Apply Qwen chat template
    # --------------------------------------------------------
    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # --------------------------------------------------------
    # 6. Tokenize and move to GPU
    # --------------------------------------------------------
    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    # --------------------------------------------------------
    # 7. Generate answer
    # --------------------------------------------------------
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    # --------------------------------------------------------
    # 8. Decode only newly generated tokens
    # --------------------------------------------------------
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "answer": answer,
        "retrieved_documents": retrieved_docs,
        "context": context
    }


print("=" * 80)
print("BASIC RAG PIPELINE CREATED")
print("=" * 80)

print("\nRetrieval: FAISS")
print("Generation: Qwen2.5-1.5B-Instruct")
print("Grounding: Knowledge-base context")
print("Pipeline status: READY")

BASIC RAG PIPELINE CREATED

Retrieval: FAISS
Generation: Qwen2.5-1.5B-Instruct
Grounding: Knowledge-base context
Pipeline status: READY


## 13. End-to-End RAG Test

The complete RAG pipeline is tested with a representative knowledge-base question.

The test verifies that the system can retrieve relevant documents from the unified vector database and generate a grounded answer using the local Qwen model.


In [48]:
# ============================================================
# End-to-End RAG Test
# ============================================================

test_question = "What is supervised learning?"

rag_result = generate_rag_answer(
    question=test_question,
    vectorstore=unified_vectorstore,
    top_k=5,
    max_new_tokens=250
)

print("=" * 80)
print("END-TO-END RAG TEST")
print("=" * 80)

print(f"\nQuestion:\n{rag_result['question']}")

print("\nGenerated Answer:")
print(rag_result["answer"])

print("\nRetrieved Sources:")

for rank, doc in enumerate(
    rag_result["retrieved_documents"],
    start=1
):
    source_type = doc.metadata.get("source_type")

    if source_type == "PDF":
        source_info = (
            f"PDF: {doc.metadata.get('source')}, "
            f"Page: {doc.metadata.get('page')}"
        )

    elif source_type == "Google Sheets":
        source_info = (
            f"Google Sheets: "
            f"Row {doc.metadata.get('row_id')}, "
            f"Topic: {doc.metadata.get('topic')}"
        )

    else:
        source_info = "Unknown source"

    print(f"{rank}. {source_info}")

END-TO-END RAG TEST

Question:
What is supervised learning?

Generated Answer:
Supervised learning is a type of machine learning where the model learns from labeled data to make predictions or decisions. The goal is to find a function \( h \) that maps input features \( x \) to output labels \( y \). This function helps predict the value of \( y \) for new, unseen data points. In supervised learning, the relationship between the input features and the output labels is known, allowing the model to generalize well beyond the training data.

Retrieved Sources:
1. Google Sheets: Row ML-001, Topic: Supervised Learning
2. PDF: cs229-notes1.pdf, Page: 2
3. PDF: cs229-notes2.pdf, Page: 1
4. Google Sheets: Row ML-007, Topic: Classification
5. Google Sheets: Row ML-011, Topic: Empirical Risk Minimization


## 14. Conversation Memory

A full conversation buffer is added to the RAG pipeline to support multi-turn interactions.

The assistant stores previous user questions and generated answers. This conversation history is included in subsequent prompts so the system can resolve references and understand follow-up questions in context.


In [49]:
# ============================================================
# Conversation Memory
# ============================================================

conversation_history = []

print("=" * 80)
print("CONVERSATION MEMORY INITIALIZED")
print("=" * 80)

print("\nMemory type: Full conversation history")
print(f"Stored messages: {len(conversation_history)}")
print("Memory status: READY")

CONVERSATION MEMORY INITIALIZED

Memory type: Full conversation history
Stored messages: 0
Memory status: READY


## 15. RAG with Conversation Memory

The RAG pipeline is extended with full conversation history.

For each new question, the system retrieves relevant knowledge from FAISS and combines it with previous conversation turns. After generating the answer, both the user question and assistant response are stored in the conversation buffer.

This enables multi-turn interactions and contextual follow-up questions.


In [50]:
# ============================================================
# RAG + Conversation Memory
# ============================================================

def generate_rag_answer_with_memory(
    question,
    vectorstore,
    top_k=5,
    max_new_tokens=250
):
    # --------------------------------------------------------
    # 1. Retrieve relevant knowledge
    # --------------------------------------------------------
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=top_k
    )

    # --------------------------------------------------------
    # 2. Build retrieved context
    # --------------------------------------------------------
    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        source_type = doc.metadata.get("source_type")

        if source_type == "PDF":
            source_info = (
                f"PDF: {doc.metadata.get('source')}, "
                f"Page: {doc.metadata.get('page')}"
            )

        elif source_type == "Google Sheets":
            source_info = (
                f"Google Sheets: "
                f"Row {doc.metadata.get('row_id')}, "
                f"Topic: {doc.metadata.get('topic')}"
            )

        else:
            source_info = "Unknown source"

        context_parts.append(
            f"[Document {i}]\n"
            f"Source: {source_info}\n"
            f"{doc.page_content}"
        )

    context = "\n\n".join(context_parts)

    # --------------------------------------------------------
    # 3. Build conversation history
    # --------------------------------------------------------
    if conversation_history:
        history_parts = []

        for message in conversation_history:
            role = message["role"].upper()
            content = message["content"]

            history_parts.append(
                f"{role}: {content}"
            )

        history_text = "\n".join(history_parts)

    else:
        history_text = "No previous conversation."

    # --------------------------------------------------------
    # 4. System instructions
    # --------------------------------------------------------
    system_message = """You are an AI Knowledge Assistant with conversation memory.

Answer the user's current question using the provided knowledge base
and the previous conversation when necessary.

Rules:
1. Use the knowledge base context as the primary source of factual information.
2. Use conversation history to understand references and follow-up questions.
3. Do not invent facts that are not supported by the knowledge base.
4. If the answer cannot be determined from the knowledge base, say:
   "I don't know based on the provided knowledge base."
5. Resolve references such as "it", "they", "this", or "the previous topic"
   using the conversation history.
6. Give a clear and concise answer.
"""

    # --------------------------------------------------------
    # 5. Build user message
    # --------------------------------------------------------
    user_message = f"""Previous Conversation:

{history_text}

Knowledge Base Context:

{context}

Current User Question:

{question}

Answer:"""

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    # --------------------------------------------------------
    # 6. Apply chat template
    # --------------------------------------------------------
    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # --------------------------------------------------------
    # 7. Tokenize
    # --------------------------------------------------------
    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    # --------------------------------------------------------
    # 8. Generate response
    # --------------------------------------------------------
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    # --------------------------------------------------------
    # 9. Decode generated tokens
    # --------------------------------------------------------
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # 10. Store conversation turn
    # --------------------------------------------------------
    conversation_history.append({
        "role": "user",
        "content": question
    })

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return {
        "question": question,
        "answer": answer,
        "retrieved_documents": retrieved_docs,
        "context": context
    }


print("=" * 80)
print("RAG + CONVERSATION MEMORY PIPELINE CREATED")
print("=" * 80)

print(f"\nMemory messages: {len(conversation_history)}")
print("Retrieval: FAISS")
print("Generation: Qwen2.5-1.5B-Instruct")
print("Memory: Full conversation buffer")
print("Pipeline status: READY")

RAG + CONVERSATION MEMORY PIPELINE CREATED

Memory messages: 0
Retrieval: FAISS
Generation: Qwen2.5-1.5B-Instruct
Memory: Full conversation buffer
Pipeline status: READY


## 16. Multi-Turn Memory Test

The memory-enabled RAG pipeline is evaluated through a sequence of related questions.

The test uses follow-up questions that depend on previous conversation turns. This verifies whether the assistant can preserve context and resolve references across multiple interactions.


In [51]:
# ============================================================
# Multi-Turn Memory Test
# ============================================================

conversation_history.clear()

memory_test_questions = [
    "What is supervised learning?",
    "What are its two common types?",
    "Which one deals with continuous target values?"
]

memory_test_results = []

for turn, question in enumerate(
    memory_test_questions,
    start=1
):
    result = generate_rag_answer_with_memory(
        question=question,
        vectorstore=unified_vectorstore,
        top_k=5,
        max_new_tokens=250
    )

    memory_test_results.append(result)

    print("=" * 80)
    print(f"TURN {turn}")
    print("=" * 80)

    print(f"\nQuestion:\n{question}")

    print("\nAnswer:")
    print(result["answer"])

    print(
        f"\nMemory messages after turn: "
        f"{len(conversation_history)}"
    )

print("\n" + "=" * 80)
print("MULTI-TURN MEMORY TEST COMPLETED")
print("=" * 80)

TURN 1

Question:
What is supervised learning?

Answer:
Supervised learning is a type of machine learning where the model learns from labeled data. The goal is to predict the correct output for new, unseen input data. In supervised learning, there is a known relationship between the input features and the desired output, which allows the model to make predictions based on patterns learned from the training data. This process involves fitting a hypothesis function to minimize the difference between the actual outcomes and the predicted ones, typically measured through loss functions.

Memory messages after turn: 2
TURN 2

Question:
What are its two common types?

Answer:
Supervised learning has two common types: classification and regression. Classification problems involve predicting discrete outputs like spam/non-spam, while regression problems deal with continuous outputs like predicting prices or scores.

Memory messages after turn: 4
TURN 3

Question:
Which one deals with continuou

## 17. Retrieval Evaluation

The semantic retrieval component is evaluated using representative questions covering multiple topics from the knowledge base.

For each query, the system retrieves the top 5 documents from FAISS. A retrieval hit is counted when the expected topic appears in the retrieved results.

The evaluation uses **Hit@5** as the retrieval metric.


In [52]:
# ============================================================
# Retrieval Evaluation — Hit@5
# ============================================================

retrieval_test_cases = [
    {
        "question": "What is supervised learning?",
        "expected_topic": "Supervised Learning"
    },
    {
        "question": "What is linear regression?",
        "expected_topic": "Linear Regression"
    },
    {
        "question": "What is logistic regression?",
        "expected_topic": "Logistic Regression"
    },
    {
        "question": "What is regularization?",
        "expected_topic": "Regularization"
    },
    {
        "question": "What is gradient descent?",
        "expected_topic": "Gradient Descent"
    },
    {
        "question": "What is a support vector machine?",
        "expected_topic": "Support Vector Machines"
    },
    {
        "question": "What is backpropagation?",
        "expected_topic": "Backpropagation"
    },
    {
        "question": "What is PCA?",
        "expected_topic": "PCA"
    },
    {
        "question": "What is K-means clustering?",
        "expected_topic": "K-Means"
    },
    {
        "question": "What is Q-learning?",
        "expected_topic": "Q-Learning"
    }
]

retrieval_results = []

for case in retrieval_test_cases:
    results = unified_vectorstore.similarity_search(
        case["question"],
        k=5
    )

    retrieved_topics = []

    for doc in results:
        topic = doc.metadata.get("topic")

        if topic:
            retrieved_topics.append(topic)

    hit = case["expected_topic"] in retrieved_topics

    retrieval_results.append({
        "question": case["question"],
        "expected_topic": case["expected_topic"],
        "hit_at_5": hit,
        "retrieved_topics": retrieved_topics
    })


hits = sum(
    result["hit_at_5"]
    for result in retrieval_results
)

total = len(retrieval_results)
hit_at_5 = (hits / total) * 100


print("=" * 80)
print("RETRIEVAL EVALUATION — HIT@5")
print("=" * 80)

for i, result in enumerate(
    retrieval_results,
    start=1
):
    status = "PASS" if result["hit_at_5"] else "MISS"

    print(
        f"\n{i}. {status}"
        f"\nQuestion: {result['question']}"
        f"\nExpected: {result['expected_topic']}"
        f"\nRetrieved Topics: {result['retrieved_topics']}"
    )

print("\n" + "=" * 80)
print(f"Hits: {hits}/{total}")
print(f"Hit@5: {hit_at_5:.2f}%")
print("=" * 80)

RETRIEVAL EVALUATION — HIT@5

1. PASS
Question: What is supervised learning?
Expected: Supervised Learning
Retrieved Topics: ['Supervised Learning', 'Classification', 'Empirical Risk Minimization']

2. PASS
Question: What is linear regression?
Expected: Linear Regression
Retrieved Topics: ['Linear Regression']

3. PASS
Question: What is logistic regression?
Expected: Logistic Regression
Retrieved Topics: ['Logistic Regression', 'Linear Regression']

4. PASS
Question: What is regularization?
Expected: Regularization
Retrieved Topics: ['Regularization']

5. PASS
Question: What is gradient descent?
Expected: Gradient Descent
Retrieved Topics: ['Gradient Descent', 'Backpropagation']

6. PASS
Question: What is a support vector machine?
Expected: Support Vector Machines
Retrieved Topics: ['Support Vector Machines', 'Soft Margin SVM', 'Maximum Margin Classifier']

7. PASS
Question: What is backpropagation?
Expected: Backpropagation
Retrieved Topics: ['Backpropagation', 'Forward Propagation', 

In [53]:
# ============================================================
# Answer Quality Evaluation — Qualitative Grounded QA
# ============================================================

answer_quality_test_cases = [
    "What is supervised learning?",
    "What is the purpose of regularization?",
    "What is gradient descent?",
    "What is backpropagation?",
    "What is Q-learning?"
]

answer_quality_results = []

print("=" * 80)
print("ANSWER QUALITY EVALUATION")
print("=" * 80)

for i, question in enumerate(answer_quality_test_cases, start=1):

    result = generate_rag_answer(
        question=question,
        vectorstore=unified_vectorstore,
        top_k=5,
        max_new_tokens=200
    )

    answer = result["answer"]
    retrieved_documents = result["retrieved_documents"]

    answer_quality_results.append({
        "question": question,
        "answer": answer,
        "retrieved_documents": retrieved_documents
    })

    print(f"\n{i}. Question: {question}")
    print(f"\nAnswer:\n{answer}")

    print("\nRetrieved Sources:")

    for j, doc in enumerate(retrieved_documents, start=1):

        source_type = doc.metadata.get("source_type")
        source = doc.metadata.get("source")
        page = doc.metadata.get("page")
        topic = doc.metadata.get("topic")
        row_id = doc.metadata.get("row_id")

        if source_type == "PDF":
            source_info = f"PDF: {source} | Page: {page}"

        elif source_type == "Google Sheets":
            source_info = (
                f"Google Sheets: {source} | "
                f"Row: {row_id} | "
                f"Topic: {topic}"
            )

        else:
            source_info = f"{source_type}: {source}"

        print(f"  {j}. {source_info}")

print("\n" + "=" * 80)
print(f"Qualitative QA cases tested: {len(answer_quality_results)}")
print("Evaluation method: Manual inspection of answer relevance and grounding")
print("=" * 80)

ANSWER QUALITY EVALUATION

1. Question: What is supervised learning?

Answer:
Supervised learning is a type of machine learning where the model learns from labeled data to make predictions or decisions. The goal is to find a function \( h \) that maps input features \( x \) to output labels \( y \). This function helps predict the value of \( y \) for new, unseen data points. In supervised learning, the relationship between the input features and the output labels is known, allowing the model to generalize well beyond the training data.

Retrieved Sources:
  1. Google Sheets: AI Knowledge Assistant - Structured Knowledge | Row: ML-001 | Topic: Supervised Learning
  2. PDF: cs229-notes1.pdf | Page: 2
  3. PDF: cs229-notes2.pdf | Page: 1
  4. Google Sheets: AI Knowledge Assistant - Structured Knowledge | Row: ML-007 | Topic: Classification
  5. Google Sheets: AI Knowledge Assistant - Structured Knowledge | Row: ML-011 | Topic: Empirical Risk Minimization

2. Question: What is the purpose

In [54]:
# ============================================================
# Out-of-Scope / Hallucination Behavior Evaluation
# ============================================================

out_of_scope_questions = [
    "What is the capital of Japan?",
    "Who won the FIFA World Cup in 2022?",
    "What is the current price of Bitcoin?"
]

out_of_scope_results = []

print("=" * 80)
print("OUT-OF-SCOPE / HALLUCINATION BEHAVIOR EVALUATION")
print("=" * 80)

for i, question in enumerate(out_of_scope_questions, start=1):

    result = generate_rag_answer(
        question=question,
        vectorstore=unified_vectorstore,
        top_k=5,
        max_new_tokens=150
    )

    answer = result["answer"]

    fallback_detected = (
        "I don't know based on the provided knowledge base."
        in answer
    )

    out_of_scope_results.append({
        "question": question,
        "answer": answer,
        "fallback_detected": fallback_detected
    })

    status = "PASS" if fallback_detected else "CHECK"

    print(f"\n{i}. {status}")
    print(f"Question: {question}")
    print(f"Answer: {answer}")

fallback_count = sum(
    result["fallback_detected"]
    for result in out_of_scope_results
)

total_out_of_scope = len(out_of_scope_results)

print("\n" + "=" * 80)
print(
    f"Fallback responses: "
    f"{fallback_count}/{total_out_of_scope}"
)
print("=" * 80)
print(
    "Interpretation: "
    "Out-of-scope questions should trigger the knowledge-base fallback "
    "instead of unsupported answers."
)
print("=" * 80)

OUT-OF-SCOPE / HALLUCINATION BEHAVIOR EVALUATION

1. PASS
Question: What is the capital of Japan?
Answer: I don't know based on the provided knowledge base.

2. PASS
Question: Who won the FIFA World Cup in 2022?
Answer: I don't know based on the provided knowledge base.

3. PASS
Question: What is the current price of Bitcoin?
Answer: I don't know based on the provided knowledge base.

Fallback responses: 3/3
Interpretation: Out-of-scope questions should trigger the knowledge-base fallback instead of unsupported answers.


# Final Project Results

## Knowledge Base

The assistant integrates knowledge from two complementary sources:

* **14 Stanford CS229 lecture-note PDFs**
* **50 structured Google Sheets knowledge records**

The PDF corpus contains:

* **157 pages**
* **49,041 extracted words**
* **355 text chunks**

The Google Sheets source contains:

* **50 structured records**
* **7 columns**
* Topics covering machine learning, deep learning, learning theory, probabilistic models, reinforcement learning, and optimization.

## Retrieval System

All knowledge was embedded using:

`sentence-transformers/all-MiniLM-L6-v2`

The embeddings were stored in a unified **FAISS** vector database.

* Embedding dimension: **384**
* PDF chunks: **355**
* Google Sheets documents: **50**
* Total indexed documents: **405**

The system performs semantic Top-K retrieval before generating an answer.

## Generation System

The retrieved knowledge is passed to:

`Qwen/Qwen2.5-1.5B-Instruct`

The model runs locally on the Colab **NVIDIA Tesla T4 GPU** using deterministic generation.

The prompting strategy instructs the model to:

* Use the provided knowledge base as the primary factual source.
* Use conversation history for follow-up questions.
* Avoid unsupported claims.
* Return a knowledge-base fallback when the requested information is unavailable.

## Conversation Memory

The assistant maintains a **full conversation buffer** containing previous user questions and assistant answers.

The memory system was tested across multiple turns, including contextual follow-up questions such as:

1. Asking about supervised learning.
2. Asking for its common types.
3. Asking which type uses continuous target values.

This demonstrated the assistant's ability to maintain conversational context and resolve references to previous discussion.

## Evaluation Results

### 1. Retrieval Evaluation

A 10-question retrieval benchmark was used to measure whether the expected structured topic appeared within the Top-5 retrieved Google Sheets results.

**Hit@5: 8/10 = 80.00%**

Two queries were marked as misses because of topic-name differences between the evaluation labels and the structured records:

* `PCA` → retrieved as `Principal Component Analysis`
* `K-Means` → retrieved as `K-Means Clustering`

This illustrates a limitation of the current evaluation method: it uses exact topic-name matching rather than semantic equivalence.

### 2. Answer Quality Evaluation

Five representative questions were evaluated through qualitative inspection.

The generated answers were checked for:

* Relevance to the question.
* Grounding in retrieved knowledge.
* Coherence.
* Presence of supporting retrieved sources.

**Result: 5 qualitative QA cases evaluated successfully.**

### 3. Out-of-Scope Behavior

Three questions outside the knowledge base were tested:

* Capital of Japan.
* 2022 FIFA World Cup winner.
* Current Bitcoin price.

All three returned the predefined knowledge-base fallback.

**Fallback responses: 3/3**

This demonstrates the intended behavior for questions that cannot be answered from the available knowledge base.

## Final System Capabilities

The completed assistant supports:

* PDF knowledge ingestion.
* Google Sheets integration.
* Structured document conversion.
* Text chunking.
* Hugging Face embeddings.
* Unified FAISS indexing.
* Semantic Top-K retrieval.
* Retrieval-Augmented Generation.
* Local LLM inference.
* Full conversation memory.
* Multi-turn follow-up handling.
* Knowledge-grounded responses.
* Out-of-scope fallback behavior.
* Retrieval and QA evaluation.

## Final Architecture

```text
                    USER QUESTION
                          │
                          ▼
                  Query Processing
                          │
                          ▼
                 FAISS Semantic Search
                          │
             ┌────────────┴────────────┐
             │                         │
             ▼                         ▼
       PDF Knowledge             Google Sheets
       14 PDFs / 157 Pages       50 Records
             │                         │
             └────────────┬────────────┘
                          │
                          ▼
                 Retrieved Context
                          │
                          ▼
               Conversation Memory
                          │
                          ▼
             Qwen2.5-1.5B-Instruct
                          │
                          ▼
                 Grounded Answer
                          │
                          ▼
                         USER
```

## Conclusion

This project demonstrates a complete local **RAG-based AI Knowledge Assistant with conversation memory**, combining heterogeneous knowledge sources, semantic retrieval, grounded generation, and multi-turn interaction in a reproducible workflow.

The evaluation confirms that the system can retrieve relevant knowledge, generate grounded answers, maintain conversational context, and safely decline questions outside its knowledge base.
